In [1]:
#Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
#Load Dataset

import bz2
import pandas as pd

# Load fastText data manually as it's not a standard CSV format
data = []
try:
    with bz2.open("/content/test.ft.txt.bz2", "rt", encoding="utf-8") as bzfile:
        for line in bzfile:
            # FastText lines are typically '__label__X some text'
            parts = line.strip().split(' ', 1) # Split only on the first space
            if len(parts) >= 2:
                label_part = parts[0]
                text_part = parts[1]

                # Extract score from label, handling potential errors in case of malformed lines
                try:
                    score = int(label_part.replace('__label__', ''))
                    data.append({'Text': text_part, 'Score': score})
                except ValueError:
                    # Skip lines that don't conform to the __label__X format
                    continue
except EOFError:
    print("Warning: Compressed file ended unexpectedly. Processing partial data.")

df = pd.DataFrame(data)

# Keep only required columns (already done by manual parsing, but good for consistency)
df = df[['Text', 'Score']]

# Convert score to sentiment - Adjusted to create at least two classes from available scores (1 and 2)
def convert_label(score):
    if score == 1:
        return 0   # Negative
    elif score == 2:
        return 1   # Positive (or less negative)
    # If the original full dataset had scores 3, 4, 5, the logic below would apply.
    # For now, assuming only 1 and 2 are prevalent in the loaded partial data.
    elif score == 3:
        return 1   # Neutral (mapped to 1 for binary classification if only 1,2 present)
    else:
        return 2   # Positive (mapped to 2 for ternary, but 1 for binary if only 1,2 present)

df['label'] = df['Score'].apply(convert_label)

# Reduce size for faster training
df = df.sample(50000, random_state=42)

print(df.head())
print(f"Distribution of labels in sampled data:\n{df['label'].value_counts()}")

                                                    Text  Score  label
32631  Several classics in season six...packaging isn...      2      1
51733  Yu-Gi-Oh! game time.: It's a good game for tho...      2      1
93276  Trike for Tot: This is a neat first trike for ...      2      1
96037  I can't believe this is the same band...: Like...      1      0
40798  superb visual execution: Stylish, smart suspen...      2      1
Distribution of labels in sampled data:
label
1    25168
0    24832
Name: count, dtype: int64


In [3]:
#TF-IDF Vectorization

# Split data into training and testing sets
X = df['Text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [4]:
#Train Model (Naive Bayes)
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

MultinomialNB()

In [5]:
#Evaluate Model
y_pred = nb_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)
print("Naive Bayes Accuracy:", accuracy)

Naive Bayes Accuracy: 0.8374


In [6]:
#Logistic Regression (Better Performance)
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train_vec, y_train)

y_pred_lr = lr_model.predict(X_test_vec)
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.876


In [7]:
#test
test_sentences = [
    "This product is amazing, I love it!",
    "Worst purchase ever, completely useless",
    "It is okay, not great but not bad",
    "Excellent quality and fast delivery",
    "Very disappointing and poor build quality"
]

test_vec = vectorizer.transform(test_sentences)
preds = lr_model.predict(test_vec)

for text, pred in zip(test_sentences, preds):
    if pred == 2:
        sentiment = "Positive"
    elif pred == 1:
        sentiment = "Neutral"
    else:
        sentiment = "Negative"

    print(f"Text: {text}")
    print(f"Prediction: {sentiment}\n")

Text: This product is amazing, I love it!
Prediction: Neutral

Text: Worst purchase ever, completely useless
Prediction: Negative

Text: It is okay, not great but not bad
Prediction: Negative

Text: Excellent quality and fast delivery
Prediction: Neutral

Text: Very disappointing and poor build quality
Prediction: Negative

